# glyphsketch: train the encoder on a Kaggle GPU

This notebook runs one M6 experiment (`glyphsketch.model.experiments`) on a GPU and writes
`glyphsketch-results-<run>.zip`, laid out like `$DATA_DIR`.

1. In the devcontainer: `cd training && uv run python -m glyphsketch.tools.kaggle_bundle`.
   This writes `$DATA_DIR/kaggle/glyphsketch-kaggle.zip`.
2. On Kaggle, create a **private** dataset from that zip. It holds data derived from
   Detexify (ODbL) and font renders, so it must not be public.
3. Import this notebook, add the dataset (**Add Input**), and set
   **Accelerator → GPU**. Internet access is not needed.
4. Check the settings in the next cell, then **Save Version → Save & Run All (Commit)**.
   The run continues in the background, and you can close the browser.
5. When it has finished, download `glyphsketch-results-<run>.zip` from the version's
   **Output** tab. Copy it into the devcontainer and unpack it with
   `unzip -o glyphsketch-results-*.zip -d /data`.

In [ ]:
EXPERIMENT = "synthetic-and-real"  # or "synthetic-only"
RUN_NAME = f"{EXPERIMENT}-kaggle"
STEPS = 40_000
BATCH_CHARACTERS = 256
BUDGET_MINUTES = 540  # the learning rate anneals to this budget; Kaggle stops runs at 12 h

In [ ]:
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

import torch

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

manifests = sorted(INPUT.rglob("MANIFEST.json"))
if not manifests:
    # Kaggle normally unpacks uploaded zips; unpack it here if it did not.
    archives = sorted(INPUT.rglob("glyphsketch-kaggle.zip"))
    if not archives:
        raise FileNotFoundError("Add the glyphsketch-kaggle dataset to this notebook (Add Input).")
    unpacked = Path("/tmp/glyphsketch-bundle")
    with zipfile.ZipFile(archives[0]) as bundle:
        bundle.extractall(unpacked)
    manifests = [unpacked / "MANIFEST.json"]
BUNDLE = manifests[0].parent
print("bundle:", BUNDLE, "| commit:", json.loads(manifests[0].read_text())["commit"])

DATA_DIR = WORK / "data"
DATA_DIR.mkdir(exist_ok=True)
for stage in sorted((BUNDLE / "data").iterdir()):
    link = DATA_DIR / stage.name
    if not link.exists():
        link.symlink_to(stage, target_is_directory=True)
ENV = dict(os.environ, DATA_DIR=str(DATA_DIR), PYTHONPATH=str(BUNDLE / "code"))

print("Python", sys.version.split()[0], "| torch", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU: set Accelerator → GPU in the notebook settings.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
command = [
    sys.executable,
    "-u",
    "-m",
    "glyphsketch.model.experiments",
    EXPERIMENT,
    "--run-name",
    RUN_NAME,
    "--steps",
    str(STEPS),
    "--batch-characters",
    str(BATCH_CHARACTERS),
    "--budget-minutes",
    str(BUDGET_MINUTES),
    "--threads",
    str(os.cpu_count()),
]
with subprocess.Popen(
    command, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
if process.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {process.returncode}")

In [ ]:
results = WORK / f"glyphsketch-results-{RUN_NAME}.zip"
with zipfile.ZipFile(results, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted((DATA_DIR / "experiments" / RUN_NAME).iterdir()):
        archive.write(path, f"experiments/{RUN_NAME}/{path.name}")
    for path in sorted((DATA_DIR / "evaluations").glob(f"encoder-{RUN_NAME}-*.json")):
        archive.write(path, f"evaluations/{path.name}")
# Keep the saved output small: drop the links to the input data.
for stage in DATA_DIR.iterdir():
    if stage.is_symlink():
        stage.unlink()
print(f"{results} ({results.stat().st_size / 1024:.0f} KB)")